In [1]:
import sqlite3

# Jouw specifieke database namen:
db_bron = 'biketodrive.db'
db_dwh = 'biketodrive_dwh.db'
sql_script = 'DWHschema.sql'

conn = sqlite3.connect(db_dwh)
cursor = conn.cursor()

print("Data Warehouse aanmaken...")
with open(sql_script, 'r') as file:
    sql_queries = file.read()
    cursor.executescript(sql_queries)

conn.commit()
conn.close()
print("✅ Data Warehouse 'biketodrive_dwh.db' aangemaakt met de juiste structuur!")

Data Warehouse aanmaken...
✅ Data Warehouse 'biketodrive_dwh.db' aangemaakt met de juiste structuur!


In [2]:
import sqlite3
import pandas as pd
from datetime import date

conn_sdm = sqlite3.connect('biketodrive.db')
conn_dwh = sqlite3.connect('biketodrive_dwh.db')
cursor_dwh = conn_dwh.cursor()
today = str(date.today())

df_klant = pd.read_sql_query("SELECT * FROM Klant", conn_sdm)
df_mapped = pd.read_sql_query("""
    SELECT source_key, dwh_surrogate FROM DWH_KeyMapping
    WHERE dwh_table='DIM_Klant' AND source_table='Klant'
""", conn_dwh)
already_loaded = set(df_mapped['source_key'])

scd2_cols = ['naam', 'adres', 'woonplaats', 'geslacht', 'geboortedatum']
inserted = updated = 0

for _, row in df_klant.iterrows():
    klantnr = row['klantnr']
    geboortedatum = str(row['geboortedatum']) if pd.notna(row.get('geboortedatum')) else None

    if klantnr not in already_loaded:
        cursor_dwh.execute("""
            INSERT INTO DIM_Klant
                (klantnr, naam, adres, woonplaats, geslacht, geboortedatum, leeftijdscategorie, geldig_van, geldig_tot, is_actief)
            VALUES (?,?,?,?,?,?,?,?, '9999-12-31', 1)
        """, (klantnr, row['naam'], row.get('adres'), row.get('woonplaats'), row.get('geslacht'), geboortedatum, None, today))

        surrogate = cursor_dwh.lastrowid
        cursor_dwh.execute("INSERT OR IGNORE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES ('DIM_Klant','Klant',?,?)", (klantnr, surrogate))
        inserted += 1
    else:
        surrogate = df_mapped.loc[df_mapped['source_key'] == klantnr, 'dwh_surrogate'].values[0]
        df_current = pd.read_sql_query("SELECT * FROM DIM_Klant WHERE klantnr=? AND is_actief=1", conn_dwh, params=(klantnr,))
        
        if not df_current.empty:
            current = df_current.iloc[0]
            changed = any(str(row.get(col, '')) != str(current.get(col, '')) for col in scd2_cols)

            if changed:
                cursor_dwh.execute("UPDATE DIM_Klant SET geldig_tot=?, is_actief=0 WHERE klantnr=? AND is_actief=1", (today, klantnr))
                cursor_dwh.execute("""
                    INSERT INTO DIM_Klant
                        (klantnr, naam, adres, woonplaats, geslacht, geboortedatum, leeftijdscategorie, geldig_van, geldig_tot, is_actief)
                    VALUES (?,?,?,?,?,?,?,?, '9999-12-31', 1)
                """, (klantnr, row['naam'], row.get('adres'), row.get('woonplaats'), row.get('geslacht'), geboortedatum, None, today))

                new_surrogate = cursor_dwh.lastrowid
                cursor_dwh.execute("UPDATE DWH_KeyMapping SET dwh_surrogate=? WHERE dwh_table='DIM_Klant' AND source_table='Klant' AND source_key=?", (new_surrogate, klantnr))
                updated += 1

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()
print(f"✅ DIM_Klant → {inserted} ingevoegd, {updated} geüpdatet (SCD2)")

✅ DIM_Klant → 0 ingevoegd, 0 geüpdatet (SCD2)


In [3]:
import sqlite3
import pandas as pd
from datetime import date

conn_sdm = sqlite3.connect('biketodrive.db')
conn_dwh = sqlite3.connect('biketodrive_dwh.db')
cursor_dwh = conn_dwh.cursor()
today = str(date.today())

def prijsklasse(prijs):
    if prijs < 50: return 'Low'
    elif prijs < 150: return 'Mid'
    else: return 'High'

def load_dim_product(df_source, source_table, productnr_col, producttype, naam_col, type_col, prijs_col, inkoop_col, kleur_col=None, merk_col=None):
    df_mapped = pd.read_sql_query(f"SELECT source_key, dwh_surrogate FROM DWH_KeyMapping WHERE dwh_table='DIM_Product' AND source_table='{source_table}'", conn_dwh)
    already_loaded = set(df_mapped['source_key'])
    inserted = updated = 0

    for _, row in df_source.iterrows():
        src_key = row.iloc[0]
        kleur   = row.get(kleur_col) if kleur_col else None
        merk    = row.get(merk_col)  if merk_col  else None
        pk      = prijsklasse(row[prijs_col])

        if src_key not in already_loaded:
            cursor_dwh.execute("""
                INSERT INTO DIM_Product (type, naam, producttype, standaardprijs, inkoopprijs, prijsklasse, kleur, merk, geldig_van, geldig_tot, is_actief)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, '9999-12-31', 1) 
            """, (row[type_col], row[naam_col], producttype, row[prijs_col], row[inkoop_col], pk, kleur, merk, today))

            surrogate = cursor_dwh.lastrowid
            cursor_dwh.execute("INSERT OR IGNORE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES ('DIM_Product',?,?,?)", (source_table, src_key, surrogate))
            inserted += 1
        else:
            surrogate = df_mapped.loc[df_mapped['source_key'] == src_key, 'dwh_surrogate'].values[0]
            df_current = pd.read_sql_query("SELECT * FROM DIM_Product WHERE productnr=? AND is_actief=1", conn_dwh, params=(int(surrogate),))
            if df_current.empty: continue
            
            current = df_current.iloc[0]
            col_mapping = {naam_col: 'naam', type_col: 'type', prijs_col: 'standaardprijs', inkoop_col: 'inkoopprijs'}
            if kleur_col: col_mapping[kleur_col] = 'kleur'
            if merk_col:  col_mapping[merk_col]  = 'merk'

            changed = any(str(row.get(src_col, '')) != str(current.get(dwh_col, '')) for src_col, dwh_col in col_mapping.items())
            if changed:
                cursor_dwh.execute("UPDATE DIM_Product SET geldig_tot=?, is_actief=0 WHERE productnr=? AND is_actief=1", (today, int(surrogate)))
                cursor_dwh.execute("""
                    INSERT INTO DIM_Product (type, naam, producttype, standaardprijs, inkoopprijs, prijsklasse, kleur, merk, geldig_van, geldig_tot, is_actief)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, '9999-12-31', 1)
                """, (row[type_col], row[naam_col], producttype, row[prijs_col], row[inkoop_col], pk, kleur, merk, today))

                new_surrogate = cursor_dwh.lastrowid
                cursor_dwh.execute("UPDATE DWH_KeyMapping SET dwh_surrogate=? WHERE dwh_table='DIM_Product' AND source_table=? AND source_key=?", (new_surrogate, source_table, src_key))
                updated += 1
    return inserted, updated

df_fiets = pd.read_sql_query("SELECT * FROM Fiets", conn_sdm)
ins_f, upd_f = load_dim_product(df_fiets, 'Fiets', 'fietsnr', 'Fiets', 'type', 'soort', 'standaardprijs', 'inkoopprijs', 'kleur', 'merk')
print(f"✅ DIM_Product (Fiets)      → {ins_f} ingevoegd, {upd_f} geüpdatet (SCD2)")

df_acc = pd.read_sql_query("SELECT * FROM Accessoire", conn_sdm)
ins_a, upd_a = load_dim_product(df_acc, 'Accessoire', 'accessoirenr', 'Accessoire', 'naam', 'soort', 'standaardprijs', 'inkoopprijs')
print(f"✅ DIM_Product (Accessoire) → {ins_a} ingevoegd, {upd_a} geüpdatet (SCD2)")

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()

✅ DIM_Product (Fiets)      → 0 ingevoegd, 0 geüpdatet (SCD2)
✅ DIM_Product (Accessoire) → 0 ingevoegd, 0 geüpdatet (SCD2)


Cell 1

In [4]:
import sqlite3
import pandas as pd

conn_sdm = sqlite3.connect('biketodrive.db')
conn_dwh = sqlite3.connect('biketodrive_dwh.db')
cursor_dwh = conn_dwh.cursor()

# Monteurs & Filialen (DIM_Medewerker)
df_monteur = pd.read_sql_query("SELECT * FROM Monteur", conn_sdm)
df_filiaal = pd.read_sql_query("SELECT * FROM Filiaal", conn_sdm)
df_monteur = df_monteur.merge(df_filiaal, left_on='filiaal', right_on='filiaalnr', suffixes=('', '_fil'))
df_mapped_m = pd.read_sql_query("SELECT source_key FROM DWH_KeyMapping WHERE dwh_table='DIM_Medewerker'", conn_dwh)
already_m = set(df_mapped_m['source_key'])

ins_m = upd_m = 0
for _, row in df_monteur.iterrows():
    monteurnr = row['monteurnr']
    if monteurnr not in already_m:
        cursor_dwh.execute("INSERT OR REPLACE INTO DIM_Medewerker (monteurID, naam, uurloon, woonplaats, filiaal, regio, provincie) VALUES (?,?,?,?,?,?,?)",
                           (monteurnr, row['naam'], row['uurloon'], row.get('woonplaats'), row.get('naam_fil'), None, row.get('provincie')))
        cursor_dwh.execute("INSERT OR IGNORE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES ('DIM_Medewerker','Monteur',?,?)", (monteurnr, monteurnr))
        ins_m += 1
    else:
        cursor_dwh.execute("UPDATE DIM_Medewerker SET naam=?, uurloon=?, woonplaats=?, filiaal=?, provincie=? WHERE monteurID=?",
                           (row['naam'], row['uurloon'], row.get('woonplaats'), row.get('naam_fil'), row.get('provincie'), monteurnr))
        upd_m += 1
print(f"✅ DIM_Medewerker → {ins_m} ingevoegd, {upd_m} geüpdatet (SCD1)")

# Partners (DIM_Partner)
def load_dim_partner(df_source, source_table, partnertype, naam_col, locatie_col):
    df_mapped = pd.read_sql_query(f"SELECT source_key, dwh_surrogate FROM DWH_KeyMapping WHERE dwh_table='DIM_Partner' AND source_table='{source_table}'", conn_dwh)
    already = set(df_mapped['source_key'])
    ins = upd = 0
    for _, row in df_source.iterrows():
        src_key = row.iloc[0]
        if src_key not in already:
            cursor_dwh.execute("INSERT OR IGNORE INTO DIM_Partner (naam, partnertype, locatie) VALUES (?,?,?)", (row[naam_col], partnertype, row.get(locatie_col)))
            surrogate = cursor_dwh.lastrowid
            cursor_dwh.execute("INSERT OR IGNORE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES ('DIM_Partner',?,?,?)", (source_table, src_key, surrogate))
            ins += 1
        else:
            surrogate = df_mapped.loc[df_mapped['source_key'] == src_key, 'dwh_surrogate'].values[0]
            cursor_dwh.execute("UPDATE DIM_Partner SET naam=?, locatie=? WHERE partnernr=?", (row[naam_col], row.get(locatie_col), int(surrogate)))
            upd += 1
    return ins, upd

df_lev = pd.read_sql_query("SELECT * FROM Leverancier", conn_sdm)
df_fab = pd.read_sql_query("SELECT * FROM Fabrikant", conn_sdm)
ins_l, upd_l = load_dim_partner(df_lev, 'Leverancier', 'Leverancier', 'naam', 'woonplaats')
ins_f, upd_f = load_dim_partner(df_fab, 'Fabrikant', 'Fabrikant', 'naam', 'plaats')
print(f"✅ DIM_Partner (Leverancier) → {ins_l} ingevoegd, {upd_l} geüpdatet")
print(f"✅ DIM_Partner (Fabrikant)   → {ins_f} ingevoegd, {upd_f} geüpdatet")

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()

✅ DIM_Medewerker → 0 ingevoegd, 15 geüpdatet (SCD1)
✅ DIM_Partner (Leverancier) → 0 ingevoegd, 5 geüpdatet
✅ DIM_Partner (Fabrikant)   → 0 ingevoegd, 11 geüpdatet


Cell 2

Cell 3

In [5]:
import sqlite3
import pandas as pd
from datetime import date

conn_sdm = sqlite3.connect('biketodrive.db')
conn_dwh = sqlite3.connect('biketodrive_dwh.db')
cursor_dwh = conn_dwh.cursor()

def ensure_datum(datum_str):
    if not datum_str or datum_str == 'None': return None
    try:
        d = pd.to_datetime(datum_str)
        datum = str(d.date())
        cursor_dwh.execute("SELECT 1 FROM DIM_Tijd WHERE datum=?", (datum,))
        if not cursor_dwh.fetchone():
            cursor_dwh.execute("INSERT INTO DIM_Tijd (datum, dag, maand, kwartaal, jaar, weekend) VALUES (?,?,?,?,?,?)",
                               (datum, d.day, d.month, (d.month - 1) // 3 + 1, d.year, 1 if d.weekday() >= 5 else 0))
        return datum
    except Exception: return None

def get_surrogate(dwh_table, source_table, source_key):
    cursor_dwh.execute("SELECT dwh_surrogate FROM DWH_KeyMapping WHERE dwh_table=? AND source_table=? AND source_key=?", (dwh_table, source_table, int(source_key)))
    row = cursor_dwh.fetchone()
    return row[0] if row else None

def al_geladen(fact_table, source_table, src_key):
    cursor_dwh.execute("SELECT 1 FROM DWH_KeyMapping WHERE dwh_table=? AND source_table=? AND source_key=?", (fact_table, source_table, int(src_key)))
    return cursor_dwh.fetchone() is not None

# FACT_Verkoop
df_fv = pd.read_sql_query("SELECT * FROM Fiets_Verkoop", conn_sdm)
df_av = pd.read_sql_query("SELECT * FROM Accessoire_Verkoop", conn_sdm)
ins_fv = ins_av = 0

for _, row in df_fv.iterrows():
    if not al_geladen('FACT_Verkoop', 'Fiets_Verkoop', row['fiets_verkoopnr']):
        datum = ensure_datum(str(row['datum']))
        prod  = get_surrogate('DIM_Product', 'Fiets', row['fiets'])
        klant = get_surrogate('DIM_Klant', 'Klant', row['klant'])
        medew = get_surrogate('DIM_Medewerker', 'Monteur', row['monteur'])
        if all([datum, prod, klant, medew]):
            omzet = row['aantal'] * row['verkoopprijs']
            cursor_dwh.execute("INSERT INTO FACT_Verkoop (productnr, klantnr, medewerkerID, datum, aantal, verkoopprijs, omzet) VALUES (?,?,?,?,?,?,?)",
                               (prod, klant, medew, datum, row['aantal'], row['verkoopprijs'], omzet))
            # FIX: INSERT OR REPLACE toegevoegd
            cursor_dwh.execute("INSERT OR REPLACE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES (?,?,?,?)", ('FACT_Verkoop', 'Fiets_Verkoop', row['fiets_verkoopnr'], cursor_dwh.lastrowid))
            ins_fv += 1

for _, row in df_av.iterrows():
    if not al_geladen('FACT_Verkoop', 'Accessoire_Verkoop', row['accessoire_verkoopnr']):
        datum = ensure_datum(str(row['datum']))
        prod  = get_surrogate('DIM_Product', 'Accessoire', row['accessoire'])
        klant = get_surrogate('DIM_Klant', 'Klant', row['klant'])
        medew = get_surrogate('DIM_Medewerker', 'Monteur', row['monteur'])
        if all([datum, prod, klant, medew]):
            omzet = row['aantal'] * row['verkoopprijs']
            cursor_dwh.execute("INSERT INTO FACT_Verkoop (productnr, klantnr, medewerkerID, datum, aantal, verkoopprijs, omzet) VALUES (?,?,?,?,?,?,?)",
                               (prod, klant, medew, datum, row['aantal'], row['verkoopprijs'], omzet))
            # FIX: INSERT OR REPLACE toegevoegd
            cursor_dwh.execute("INSERT OR REPLACE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES (?,?,?,?)", ('FACT_Verkoop', 'Accessoire_Verkoop', row['accessoire_verkoopnr'], cursor_dwh.lastrowid))
            ins_av += 1

# FACT_Inkoop
df_fi = pd.read_sql_query("SELECT * FROM Fiets_Inkoop", conn_sdm)
df_ai = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", conn_sdm)
df_fiets_all = pd.read_sql_query("SELECT fietsnr, fabrikant, inkoopprijs FROM Fiets", conn_sdm)
df_acc_all = pd.read_sql_query("SELECT accessoirenr, leverancier, inkoopprijs FROM Accessoire", conn_sdm)
ins_fi = ins_ai = 0

for _, row in df_fi.iterrows():
    if not al_geladen('FACT_Inkoop', 'Fiets_Inkoop', row['inkoopnr']):
        datum = ensure_datum(f"{int(row['inkoopjaar'])}-{int(row['inkoopmaand']):02d}-01")
        prod = get_surrogate('DIM_Product', 'Fiets', row['fiets'])
        fiets_info = df_fiets_all[df_fiets_all['fietsnr'] == row['fiets']]
        if not fiets_info.empty and prod and datum:
            partner = get_surrogate('DIM_Partner', 'Fabrikant', fiets_info.iloc[0]['fabrikant'])
            if partner:
                inkoopwaarde = row['aantal'] * fiets_info.iloc[0]['inkoopprijs']
                cursor_dwh.execute("INSERT INTO FACT_Inkoop (productnr, partnernr, datum, aantal, inkoopwaarde) VALUES (?,?,?,?,?)", (prod, partner, datum, row['aantal'], inkoopwaarde))
                # FIX: INSERT OR REPLACE toegevoegd
                cursor_dwh.execute("INSERT OR REPLACE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES (?,?,?,?)", ('FACT_Inkoop', 'Fiets_Inkoop', row['inkoopnr'], cursor_dwh.lastrowid))
                ins_fi += 1

for _, row in df_ai.iterrows():
    if not al_geladen('FACT_Inkoop', 'Accessoire_Inkoop', row['inkoopnr']):
        datum = ensure_datum(f"{int(row['inkoopjaar'])}-{int(row['inkoopmaand']):02d}-01")
        prod = get_surrogate('DIM_Product', 'Accessoire', row['accessoire'])
        acc_info = df_acc_all[df_acc_all['accessoirenr'] == row['accessoire']]
        if not acc_info.empty and prod and datum:
            partner = get_surrogate('DIM_Partner', 'Leverancier', acc_info.iloc[0]['leverancier'])
            if partner:
                inkoopwaarde = row['aantal'] * acc_info.iloc[0]['inkoopprijs']
                cursor_dwh.execute("INSERT INTO FACT_Inkoop (productnr, partnernr, datum, aantal, inkoopwaarde) VALUES (?,?,?,?,?)", (prod, partner, datum, row['aantal'], inkoopwaarde))
                # FIX: INSERT OR REPLACE toegevoegd
                cursor_dwh.execute("INSERT OR REPLACE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES (?,?,?,?)", ('FACT_Inkoop', 'Accessoire_Inkoop', row['inkoopnr'], cursor_dwh.lastrowid))
                ins_ai += 1

# FACT_Onderhoud
df_oh = pd.read_sql_query("SELECT * FROM Onderhoud", conn_sdm)
df_fv_klant = pd.read_sql_query("SELECT fiets, klant FROM Fiets_Verkoop", conn_sdm)
ins_oh = 0

for _, row in df_oh.iterrows():
    if not al_geladen('FACT_Onderhoud', 'Onderhoud', row['onderhoudnr']):
        datum = ensure_datum(str(row['datum']))
        medew = get_surrogate('DIM_Medewerker', 'Monteur', row['monteur'])
        if datum and medew:
            klant_match = df_fv_klant[df_fv_klant['fiets'] == row['fiets']]
            klant = get_surrogate('DIM_Klant', 'Klant', klant_match.iloc[0]['klant']) if not klant_match.empty else None
            
            cursor_dwh.execute("INSERT INTO FACT_Onderhoud (medewerkerID, klantnr, datum, begintijd, eindtijd) VALUES (?,?,?,?,?)",
                               (medew, klant, datum, str(row['starttijd']), str(row['eindtijd'])))
            # FIX: INSERT OR REPLACE toegevoegd
            cursor_dwh.execute("INSERT OR REPLACE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate) VALUES (?,?,?,?)", ('FACT_Onderhoud', 'Onderhoud', row['onderhoudnr'], cursor_dwh.lastrowid))
            ins_oh += 1

print(f"✅ FACT_Verkoop  → {ins_fv} Fiets, {ins_av} Accessoire")
print(f"✅ FACT_Inkoop   → {ins_fi} Fiets, {ins_ai} Accessoire")
print(f"✅ FACT_Onderhoud → {ins_oh} ingevoegd")

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()
print("✅ ETL volledig afgerond.")

✅ FACT_Verkoop  → 0 Fiets, 0 Accessoire
✅ FACT_Inkoop   → 100 Fiets, 50 Accessoire
✅ FACT_Onderhoud → 0 ingevoegd
✅ ETL volledig afgerond.


In [8]:
import sqlite3
import pandas as pd
from datetime import date

# Verbind met de databases
conn_sdm = sqlite3.connect('biketodrive.db')
conn_dwh = sqlite3.connect('biketodrive_dwh.db')
vandaag = str(date.today())

print("--- 1. VOORAF (Wat zit er nu in het DWH?) ---")
display(pd.read_sql_query("SELECT surrogate_id, klantnr, adres, geldig_van, geldig_tot, is_actief FROM DIM_Klant WHERE klantnr = 1", conn_dwh))


print("\n--- 2. BRON AANPASSEN (De klant verhuist in het 'echte' systeem) ---")
conn_sdm.execute("UPDATE Klant SET adres = 'De Allernieuwste Straat 9' WHERE klantnr = 1")
conn_sdm.commit()
print("✅ Adres succesvol aangepast in biketodrive.db")


print("\n--- 3. SCD TYPE 2 TOEPASSEN (DWH bijwerken) ---")
# A. Oude regel afsluiten (zet is_actief op 0 en vul geldig_tot in)
conn_dwh.execute(f"UPDATE DIM_Klant SET geldig_tot = '{vandaag}', is_actief = 0 WHERE klantnr = 1 AND is_actief = 1")

# B. Haal de nieuwe gegevens op uit de bron en voeg een nieuwe actieve regel toe
nieuwe_klant = pd.read_sql_query("SELECT * FROM Klant WHERE klantnr = 1", conn_sdm).iloc[0]
conn_dwh.execute("""
    INSERT INTO DIM_Klant (klantnr, naam, adres, woonplaats, geslacht, geboortedatum, geldig_van, geldig_tot, is_actief)
    VALUES (?, ?, ?, ?, ?, ?, ?, '9999-12-31', 1)
""", (1, nieuwe_klant['naam'], nieuwe_klant['adres'], nieuwe_klant['woonplaats'], nieuwe_klant['geslacht'], nieuwe_klant['geboortedatum'], vandaag))

# C. Vertel het script dat Klant 1 vanaf nu een nieuw DWH_ID (surrogate_id) heeft
laatste_id = conn_dwh.execute("SELECT MAX(surrogate_id) FROM DIM_Klant").fetchone()[0]
conn_dwh.execute("UPDATE DWH_KeyMapping SET dwh_surrogate = ? WHERE dwh_table = 'DIM_Klant' AND source_key = 1", (laatste_id,))
conn_dwh.commit()
print("✅ SCD2 is toegepast: Oude regel bewaard als historie, nieuwe regel staat op actief.")


print("\n--- 4. ACHTERAF (Wat zien we nu in het DWH?) ---")
display(pd.read_sql_query("SELECT surrogate_id, klantnr, adres, geldig_van, geldig_tot, is_actief FROM DIM_Klant WHERE klantnr = 1", conn_dwh))

# Verbindingen sluiten
conn_sdm.close()
conn_dwh.close()

--- 1. VOORAF (Wat zit er nu in het DWH?) ---


,surrogate_id,klantnr,adres,geldig_van,geldig_tot,is_actief
0,1,1,Kerkstraat 12,2026-04-07,2026-04-07,0
1,26,1,De Allernieuwste Straat 99,2026-04-07,2026-04-07,0
2,27,1,De Allernieuwste Straat 9,2026-04-07,9999-12-31,1



--- 2. BRON AANPASSEN (De klant verhuist in het 'echte' systeem) ---
✅ Adres succesvol aangepast in biketodrive.db

--- 3. SCD TYPE 2 TOEPASSEN (DWH bijwerken) ---
✅ SCD2 is toegepast: Oude regel bewaard als historie, nieuwe regel staat op actief.

--- 4. ACHTERAF (Wat zien we nu in het DWH?) ---


,surrogate_id,klantnr,adres,geldig_van,geldig_tot,is_actief
0,1,1,Kerkstraat 12,2026-04-07,2026-04-07,0
1,26,1,De Allernieuwste Straat 99,2026-04-07,2026-04-07,0
2,27,1,De Allernieuwste Straat 9,2026-04-07,2026-04-07,0
3,28,1,De Allernieuwste Straat 9,2026-04-07,9999-12-31,1
